## 10.0 Introduction

Lecture 9 gave us the mathematical foundation: the least squares formula $\hat{\mathbf{x}} = (A^TA)^{-1}A^T\mathbf{b}$ finds the best approximate solution to $A\mathbf{x} = \mathbf{b}$ when no exact solution exists. This lecture builds directly on that foundation in three steps.

First, we translate from the linear algebra notation of Lecture 9 into the **regression model** notation used in statistics and machine learning. The change is mostly cosmetic — $A$ becomes $\mathbf{X}$, $\mathbf{x}$ becomes $\boldsymbol{\beta}$, $\mathbf{b}$ becomes $\mathbf{y}$ — but it comes with one important addition: an intercept term $\beta_0$ that frees the model from being forced through the origin. We'll also introduce $R^2$, the standard metric for measuring how much of the variation in $\mathbf{y}$ a model explains.

Second, we learn to train regression models using **scikit-learn**, the dominant machine learning library in Python. In practice, practitioners almost never invert a matrix by hand. Understanding why the formula works, as we did in Lecture 9, is essential — but using a well-engineered library to apply it is how real work gets done.

Third, we ask a question that turns out to be surprisingly deep: *what is scikit-learn's `.fit()` method actually doing?* The answer connects least squares to calculus-based optimization and introduces **gradient descent** — the algorithm behind virtually every large-scale machine learning model in use today. We close by applying gradient descent ideas to a classification problem: predicting whether a baby's mother smoked during pregnancy using **logistic regression**.

Throughout the lecture we work with the `babies` dataset, which records birth outcomes and maternal health data for over 1,200 births. We'll predict birth weight (`bwt`) with linear regression, then predict smoking status (`smoke`) with logistic regression.

By the end of this lecture you will be able to:

- Construct a design matrix $\mathbf{X}$ with an intercept column and explain why the intercept matters
- Compute $\boldsymbol{\hat{\beta}}$ from scratch and interpret each weight
- Compute SSR, RMSE, and $R^2$ and explain what each one measures
- Use the scikit-learn API to train a linear regression model and make predictions
- Explain why `.fit()` sometimes uses gradient descent instead of the normal equations
- Describe the gradient descent update rule and explain the role of the learning rate
- Explain why least squares cannot be used for logistic regression and train a logistic model with scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 10.1 From Least Squares to Linear Regression

### The Regression Model

A **linear regression model** predicts a numerical output $y$ from $p$ input variables $x_1, x_2, \ldots, x_p$:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p + \epsilon$$

This is the same $A\mathbf{x} = \mathbf{b}$ structure from Lecture 9, just dressed in statistics notation. The translation is:

| Lecture 9 | Lecture 10 | Meaning |
|:---------:|:----------:|:-------|
| $A$ | $\mathbf{X}$ | the feature matrix (now called the **design matrix**) |
| $\mathbf{x}$ | $\boldsymbol{\beta}$ | the unknown weights (now called **parameters** or **coefficients**) |
| $\mathbf{b}$ | $\mathbf{y}$ | the target vector (the **response variable**, AKA "**labels**") |
| $\hat{\mathbf{b}}$ | $\hat{\mathbf{y}}$ | the predicted values |
| $\boldsymbol{\epsilon}$ | $\boldsymbol{\epsilon}$ | the **residuals** |

The variables $x_1, \ldots, x_p$ are the inputs — often called **regressors**, **features**, or **predictors**. The variable $y$ is the output — often called the **response** or **dependent variable**.

### A Key Difference between A and X: the Intercept $\beta_0$

The one meaningful addition over Lecture 9 is the intercept term $\beta_0$. In Lecture 9, our model was $A\hat{\mathbf{x}} \approx \mathbf{b}$ with no constant term — the model was constrained to pass through the origin. A regression model adds a column of all ones to the design matrix, giving $\beta_0$ a place to live:

$$\mathbf{X} = \begin{bmatrix} 1 & x_{11} & \cdots & x_{1p} \\ 1 & x_{21} & \cdots & x_{2p} \\ \vdots & \vdots & & \vdots \\ 1 & x_{n1} & \cdots & x_{np} \end{bmatrix}, \qquad \boldsymbol{\beta} = \begin{bmatrix} \beta_0 \\ \beta_1 \\ \vdots \\ \beta_p \end{bmatrix}$$

The ones column means $\mathbf{X}\boldsymbol{\beta}$ now includes the term $1 \cdot \beta_0$ in every row — a vertical shift that lets the model's predictions be centered anywhere, not just at zero. Geometrically, without $\beta_0$ the fitted line is forced through the origin; with $\beta_0$, it can cross the $y$-axis wherever the data demands.

The design matrix $\mathbf{X}$ is $n \times (p+1)$ — $n$ observations, $p$ features plus the intercept column.

### The Least Squares Solution

The least squares solution carries over unchanged:

$$\boldsymbol{\hat{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

The model's predictions are $\hat{\mathbf{y}} = \mathbf{X}\boldsymbol{\hat{\beta}}$, and the residuals are $\boldsymbol{\epsilon} = \mathbf{y} - \hat{\mathbf{y}}$. Notice that the fitted model uses $\hat{\boldsymbol\beta}$ instead of $\boldsymbol\beta$, and has no $\epsilon$ term:

$$\hat{y} = \hat{\beta}_0 + \hat{\beta}_1 x_1 + \hat{\beta}_2 x_2 + \cdots + \hat{\beta}_p x_p$$

The $\epsilon$ in the full model $y = \mathbf{X}\boldsymbol{\beta} + \epsilon$ represents everything the model can never capture — irreducible noise, unmeasured variables, randomness. The fitted model $\hat{\mathbf{y}} = \mathbf{X}\boldsymbol{\hat{\beta}}$ drops $\epsilon$ because we're computing predictions, not describing reality.

### The Babies Dataset

The `babies` dataset records birth weight and health information for 1,236 births. The variables are:

| Variable | Description |
|:--------:|:------------|
| `bwt` | Birth weight (oz) — the response variable for regression |
| `gestation` | Length of pregnancy (days) |
| `parity` | 0 if firstborn, 1 otherwise |
| `age` | Mother's age (years) |
| `height` | Mother's height (inches) |
| `weight` | Mother's pre-pregnancy weight (lbs) |
| `smoke` | 1 if mother smoked during pregnancy, 0 otherwise — the target for logistic regression in Section 10.6 |

In [ ]:
url = 'https://raw.githubusercontent.com/ajorgen1/Linear-Algebra-With-Applications/main/datasets/babies.csv'
babies = pd.read_csv(url)
babies = babies.fillna(babies.mean(numeric_only=True))   # replace NaN with column means
babies.head()

We want to predict birth weight (`bwt`) from the other five variables. Our model is:

$$\widehat{bwt} = \hat{\beta}_0 + \hat{\beta}_1\,\text{gestation} + \hat{\beta}_2\,\text{parity} + \hat{\beta}_3\,\text{age} + \hat{\beta}_4\,\text{height} + \hat{\beta}_5\,\text{weight} + \hat{\beta}_6\,\text{smoke}$$

The design matrix $\mathbf{X}$ is $n \times 7$ — a column of ones for the intercept, then six feature columns. The target vector $\mathbf{y}$ is $n \times 1$.

In [ ]:
feature_cols = ['gestation', 'parity', 'age', 'height', 'weight', 'smoke']

# Build the design matrix: column of ones on the left, then the features
X_features = babies[feature_cols].to_numpy()
ones        = np.ones((X_features.shape[0], 1))
X           = np.concatenate([ones, X_features], axis=1)
y           = babies['bwt'].to_numpy().reshape(-1, 1)

print(f'Design matrix X:  shape {X.shape}  (n={X.shape[0]} observations, p+1={X.shape[1]} columns)')
print(f'Target vector y:  shape {y.shape}')
print()
print('First three rows of X (intercept column, then features):')
print(np.round(X[:3], 2))

**Question.** The intercept column is all ones. Verify by hand that for row 0, the prediction $\hat{y}_0 = \beta_0 \cdot 1 + \beta_1 \cdot x_{01} + \ldots$ includes $\beta_0$ exactly once, regardless of the other feature values. Why couldn't we have just used any constant other than 1 in that column?

In [ ]:
# Compute the least squares solution from scratch using the normal equations
beta_hat = ???

print('Estimated coefficients beta_hat:')
labels = ['intercept'] + feature_cols
for name, val in zip(labels, beta_hat.flatten()):
    print(f'  {name:<12}: {val:+.4f}')

In [ ]:
from IPython.display import display, Markdown

b = beta_hat.flatten()
display(Markdown(
    f"$$\\widehat{{bwt}} = {b[0]:.2f} "
    f"{b[1]:+.4f}\\,\text{{gestation}} "
    f"{b[2]:+.4f}\\,\text{{parity}} "
    f"{b[3]:+.4f}\\,\text{{age}} "
    f"{b[4]:+.4f}\\,\text{{height}} "
    f"{b[5]:+.4f}\\,\text{{weight}} "
    f"{b[6]:+.4f}\\,\text{{smoke}}$$"
))

**Exercise.** The coefficient on `smoke` is negative. Does this make sense medically? What does it mean in terms of the model's predictions?

In [ ]:
# Make a prediction for a specific baby
# gestation=280 days, firstborn (parity=0), mother age=27, height=64in, weight=120lb, non-smoker
new_baby = np.array([[1, 280, 0, 27, 64, 120, 0]])
y_hat_new = np.dot(new_baby, beta_hat)

print(f'Predicted birth weight: {y_hat_new[0, 0]:.2f} oz  ({y_hat_new[0, 0] / 16:.2f} lbs)')

**Question.** The prediction above uses a dot product: `np.dot(new_baby, beta_hat)`. How does this connect to the column-view of matrix-vector multiplication from Lecture 7? What is each term in that dot product computing?

## 10.2 Model Metrics: SSR, RMSE, and $R^2$

Having fitted the model, we need to measure how well it fits. Three metrics are standard in the regression setting:

$$\text{SSR} = \|\mathbf{y} - \hat{\mathbf{y}}\|^2 = \|\boldsymbol{\epsilon}\|^2 = \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

$$\text{RMSE} = \sqrt{\frac{\text{SSR}}{n}} = \frac{\|\boldsymbol{\epsilon}\|}{\sqrt{n}}$$

$$R^2 = \frac{\text{TSS} - \text{SSR}}{\text{TSS}}, \qquad \text{where} \quad \text{TSS} = \sum_{i=1}^n (y_i - \bar{y})^2$$

**SSR** (Sum of Squared Residuals) is the raw objective function we minimized. Smaller is better, but it grows with $n$, making comparison across datasets difficult.

**RMSE** normalizes by $n$ and square-roots to restore the original units of $y$. An RMSE of 10 oz means the model is off by about 10 oz on average — directly interpretable.

**TSS** (Total Sum of Squares) measures the total variation in $\mathbf{y}$ relative to its mean. It answers: "how much variation is there to explain?"

$\mathbf{R^2}$ is the fraction of that variation the model explains. $R^2 = 1$ means a perfect fit; $R^2 = 0$ means the model does no better than simply predicting the mean $\bar{y}$ for every observation. Values close to 1 are good; values close to 0 mean the features carry little predictive information about $y$.

**Takeaway.** SSR, RMSE, and $R^2$ all boil down to the same thing: how long is $\boldsymbol{\epsilon}$? A small $\|\boldsymbol{\epsilon}\|$ means $\hat{\mathbf{y}}$ is close to $\mathbf{y}$, which means $\hat{\mathbf{y}}$ is "close" to $C(\mathbf{X})$ — the model captures most of the variation in the target.

In [ ]:
y_hat   = ???
epsilon = ???
n       = ???

SSR  = ???
RMSE = ???
TSS  = ???
R2   = ???

print(f'SSR  = {SSR:.2f} oz^2')
print(f'RMSE = {RMSE:.4f} oz   (roughly the average prediction error per prediced value)')
print(f'TSS  = {TSS:.2f} oz^2')
print(f'R^2  = {R2:.4f}   (the model explains {R2*100:.2f}% of the variation in bwt)')

In [ ]:
# Visualize fit quality
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(y, y_hat, color='steelblue', alpha=0.4, edgecolors='white', lw=0.4, s=20)
lims = [y.min() - 5, y.max() + 5]
ax.plot(lims, lims, 'k--', lw=1, alpha=0.7, label='Perfect prediction')
ax.set_xlabel('Actual birth weight (oz)')
ax.set_ylabel('Predicted birth weight (oz)')
ax.set_title(r'Actual vs. Predicted: $\mathbf{y}$ vs. $\hat{\mathbf{y}}$')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

ax = axes[1]
ax.hist(epsilon.flatten(), bins=30, color='tomato', edgecolor='white', lw=0.4)
ax.axvline(0, color='black', lw=1.2, linestyle='--')
ax.set_xlabel(r'Residual $\epsilon_i = y_i - \hat{y}_i$ (oz)')
ax.set_ylabel('Count')
ax.set_title(r'Distribution of Residuals $\boldsymbol{\epsilon}$')
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle(f'Babies Linear Regression — RMSE = {RMSE:.2f} oz,  $R^2$ = {R2:.4f}', fontsize=12)
plt.tight_layout()
plt.show()

**Question.** Suppose you built a second model that only used `gestation` as a feature. Would its $R^2$ be higher or lower than the full model? Would its SSR be higher or lower? How about its RMSE?

## 10.3 Regression with the scikit-learn API

In practice, no one computes $(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ by hand — they use a library. **scikit-learn** is the standard machine learning library in Python. Its design follows a consistent pattern across every model type:

1. **Instantiate** a model object: `model = LinearRegression()`
2. **Fit** the model to data: `model.fit(X, y)` — this is where training happens
3. **Predict** on new data: `model.predict(X_new)`
4. **Evaluate** the fit: `model.score(X, y)` returns $R^2$

One important difference: scikit-learn handles the intercept column automatically. You pass in your raw feature matrix (no column of ones), and scikit-learn adds the intercept internally. So **`X` passed to sklearn is $n \times p$**, not $n \times (p+1)$. The fitted intercept is stored in `model.intercept_` and the feature coefficients in `model.coef_`.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn import metrics

# sklearn expects X to be n x p (no intercept column) and y to be a 1D array
X_sk = babies[feature_cols].to_numpy()
y_sk = babies['bwt'].to_numpy()

babies_lm = ???

print('Intercept (beta_0):  ', round(babies_lm.intercept_, 4))
print('Coefficients (beta_1 ... beta_6):')
for name, val in zip(feature_cols, babies_lm.coef_):
    print(f'  {name:<12}: {val:+.4f}')

**Question.** Compare these coefficients to the ones we computed from scratch in Section 10.1. Are they the same? They should be — sklearn's `LinearRegression` on a dataset of this size solves the same least squares equations we derived in Lecture 9.

In [ ]:
# Two ways to compute y_hat:
# (a) using sklearn's .predict()
y_hat_sk = ???

# (b) using matrix-vector multiplication (with the full X including intercept)
y_hat_mv = ???

print('Max difference between .predict() and matrix-vector multiplication:')
print(np.max(np.abs(y_hat_sk - y_hat_mv.flatten())))

In [ ]:
# Metrics via sklearn's function
R2_sk   = ???
RMSE_sk = ???

print(f'R^2  (sklearn): {R2_sk:.4f}')
print(f'RMSE (sklearn): {RMSE_sk:.4f} oz')
print()
print(f'R^2  (from scratch): {R2:.4f}')
print(f'RMSE (from scratch): {RMSE:.4f} oz')

**Question.** `LinearRegression` has a parameter `fit_intercept` that defaults to `True`. What do you think happens if you set it to `False` and pass in the same `X_sk` — a matrix with no intercept column? What would be different about the model? How would you expect $R^2$ and RMSE to change?

**Question.** We filled NaN values with column means before fitting. This is one common approach. What are the consequences of filling with the mean rather than dropping those rows entirely? Under what circumstances might dropping rows be preferred?

## 10.4 Least Squares as an Optimization Problem

### What is `.fit()` actually doing?

For the babies dataset, sklearn computed the same equations we used from scratch. But for many real-world problems, inverting $(\mathbf{X}^T\mathbf{X})$ directly is not feasible:

- Computing a matrix inverse has computational cost $O((p+1)^3)$. A model with one million parameters — not unusual in modern machine learning — would require $10^{18}$ operations. That is not happening.
- Some models, like logistic regression (Section 10.6), are not linear in their parameters. Setting $\nabla f = \mathbf{0}$ and solving algebraically leads to the same computational dead end.
- Neural networks have millions to billions of parameters and nonlinear structures where no closed-form solution exists at all.

In all of these cases, sklearn uses an iterative algorithm called **gradient descent** to find $\boldsymbol{\beta}^*$ without ever computing an inverse.

To understand gradient descent we first need to see least squares as an optimization problem — finding the minimum of a function.

### The Loss Function

Think of SSR as a function of the parameters:

$$f(\beta_0, \beta_1, \ldots, \beta_p) = \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2 = \text{SSR}$$

This is our model's **loss function** — a single number that measures how bad the current parameters are. Training a model means finding the $\boldsymbol{\beta}^*$ that minimizes $f$. We call $f$ a "loss" because we want to minimize losses (errors).

For linear regression with two parameters $\beta_0$ and $\beta_1$, we can plot $f(\beta_0, \beta_1)$ as a surface in three dimensions. The minimum of that surface is exactly the least squares solution $\boldsymbol{\hat{\beta}}$.

In [ ]:
# @title Loss surface for a simple 1-feature model: bwt ~ beta_0 + beta_1 * gestation
from mpl_toolkits import mplot3d

X_simple = np.concatenate([np.ones((n, 1)), babies[['gestation']].to_numpy()], axis=1)
y_arr    = y.flatten()

# Compute the least squares solution for this simple model
beta_simple = np.linalg.inv(X_simple.T @ X_simple) @ X_simple.T @ y
b0_star, b1_star = beta_simple.flatten()

# Grid of (beta_0, beta_1) values centered on the optimal solution
b0_range = np.linspace(b0_star - 40, b0_star + 40, 200)
b1_range = np.linspace(b1_star - 0.3, b1_star + 0.3, 200)
B0, B1   = np.meshgrid(b0_range, b1_range)

# Evaluate the loss at each grid point
Z = np.zeros_like(B0)
for i in range(B0.shape[0]):
    for j in range(B0.shape[1]):
        beta_ij = np.array([B0[i, j], B1[i, j]])
        eps_ij  = y_arr - X_simple @ beta_ij
        Z[i, j] = np.sum(eps_ij**2)

fig = plt.figure(figsize=(13, 5))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(B0, B1, Z, cmap='Blues', alpha=0.85, linewidth=0)
ax1.scatter(b0_star, b1_star, np.sum((y_arr - X_simple @ beta_simple.flatten())**2),
            color='tomato', s=80, zorder=5, label=r'$\hat{\boldsymbol{\beta}}$ (minimum)')
ax1.set_xlabel(r'$\beta_0$', fontsize=10)
ax1.set_ylabel(r'$\beta_1$', fontsize=10)
ax1.set_zlabel('SSR', fontsize=10)
ax1.set_title('Loss surface (3D view)', fontsize=11)
ax1.view_init(elev=28, azim=220)
ax1.legend(fontsize=9)

# Contour plot (top-down view)
ax2 = fig.add_subplot(122)
cp  = ax2.contourf(B0, B1, Z, levels=40, cmap='Blues')
ax2.contour(B0, B1, Z, levels=20, colors='white', linewidths=0.5, alpha=0.5)
ax2.scatter(b0_star, b1_star, color='tomato', s=80, zorder=5,
            label=r'$\hat{\boldsymbol{\beta}}$ (minimum)')
ax2.set_xlabel(r'$\beta_0$', fontsize=11)
ax2.set_ylabel(r'$\beta_1$', fontsize=11)
ax2.set_title('Contour view (top-down)', fontsize=11)
ax2.legend(fontsize=9)
plt.colorbar(cp, ax=ax2, label='SSR')

plt.suptitle(r'Loss Function $f(\beta_0, \beta_1) = \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2$'
             '  —  Simple Model: bwt ~ gestation', fontsize=11)
plt.tight_layout()
plt.show()

print(f'Optimal parameters: beta_0* = {b0_star:.4f},  beta_1* = {b1_star:.4f}')

**Question.** The loss surface is shaped like a bowl — it has a single global minimum and no local minima. This is a special property of the squared error loss for linear models. Why does it matter for optimization? What would go wrong if the surface had multiple bowls?

**Question.** The contour lines are ellipses, not circles. What does the elongated shape of the ellipses tell you about how sensitive the loss is to changes in $\beta_0$ vs. changes in $\beta_1$? Keep this in mind for Section 10.5.

### Partial Derivatives and the Gradient

To find the minimum of $f$ we need calculus. For a function of several variables the relevant tool is the **partial derivative**.

**Definition.** The **partial derivative** of $f(\beta_0, \beta_1, \ldots, \beta_p)$ with respect to $\beta_i$ is written $\frac{\partial f}{\partial \beta_i}$ and computed by treating every variable except $\beta_i$ as a constant and differentiating normally.

**Definition.** The **gradient** $\nabla f$ is the vector of all partial derivatives:

$$\nabla f = \begin{bmatrix} \frac{\partial f}{\partial \beta_0} \\ \frac{\partial f}{\partial \beta_1} \\ \vdots \\ \frac{\partial f}{\partial \beta_p} \end{bmatrix}$$

**Key property.** At any point on the surface, $\nabla f$ points in the direction of **steepest ascent** — the direction along which $f$ increases fastest. Consequently, $-\nabla f$ points in the direction of **steepest descent**.

**Example.** For the simple one-feature model with $f(\beta_0, \beta_1) = \sum_{i=1}^n (y_i - \beta_0 - \beta_1 x_i)^2$, expanding and differentiating gives:

$$\frac{\partial f}{\partial \beta_0} = -2\sum_{i=1}^n (y_i - \beta_0 - \beta_1 x_i) = -2\mathbf{1}^T\boldsymbol{\epsilon}$$

$$\frac{\partial f}{\partial \beta_1} = -2\sum_{i=1}^n x_i(y_i - \beta_0 - \beta_1 x_i) = -2\mathbf{x}_1^T\boldsymbol{\epsilon}$$

In matrix form: $\nabla f = -2\mathbf{X}^T\boldsymbol{\epsilon} = -2\mathbf{X}^T(\mathbf{y} - \mathbf{X}\boldsymbol{\beta})$.

Setting $\nabla f = \mathbf{0}$ and rearranging gives exactly the normal equations $\mathbf{X}^T\mathbf{X}\boldsymbol{\hat{\beta}} = \mathbf{X}^T\mathbf{y}$ from Lecture 9. The calculus and linear algebra routes lead to the same place — but the calculus route suggests an alternative when solving that system is too expensive.

## 10.5 Gradient Descent

The idea of gradient descent is simple: if $\nabla f$ always points uphill, then taking a small step in the direction $-\nabla f$ always moves downhill. Repeat this process from any starting point and you will eventually reach the minimum.

### The Update Rule

Starting from an initial guess $\boldsymbol{\beta}^{(0)}$, gradient descent iterates:

$$\boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} - \alpha \nabla f\big(\boldsymbol{\beta}^{(t)}\big)$$

Here $\alpha > 0$ is the **learning rate** — a small scalar that controls the step size. At each iteration we compute the gradient at the current location, then move a little bit in the downhill direction. We stop when the gradient is close enough to $\mathbf{0}$ — meaning we've found a point where no direction leads downhill, i.e., the minimum.

For our loss function $f = \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2$, the update rule becomes:

$$\boldsymbol{\beta}^{(t+1)} = \boldsymbol{\beta}^{(t)} + 2\alpha\,\mathbf{X}^T(\mathbf{y} - \mathbf{X}\boldsymbol{\beta}^{(t)})$$

Notice that each update only requires a matrix-vector product $\mathbf{X}^T\boldsymbol{\epsilon}^{(t)}$ — no inversion, no $(p+1)^3$ cost. That is the whole point.

### The Learning Rate

Choosing $\alpha$ is critical:

- **Too large:** the steps overshoot the minimum and the algorithm diverges — the loss goes up instead of down.
- **Too small:** the steps are so tiny that convergence takes an impractical number of iterations.
- **Just right:** the loss decreases steadily and the algorithm converges to $\boldsymbol{\hat{\beta}}$ in a reasonable number of steps.

In [ ]:
# Run this cell without modification
import warnings

def gradient_descent(X, y, alpha, n_iters, beta_init=None):
    """Run gradient descent and return (beta_history, loss_history)."""
    n, p = X.shape
    beta   = np.zeros(p) if beta_init is None else beta_init.copy()
    betas  = [beta.copy()]
    losses = [float(np.sum((y - X @ beta)**2))]
    for _ in range(n_iters):
        eps  = y - X @ beta
        grad = -2 * X.T @ eps
        beta = beta - alpha * grad
        betas.append(beta.copy())
        loss = float(np.sum((y - X @ beta)**2))
        losses.append(loss if np.isfinite(loss) else np.inf)
    return np.array(betas), np.array(losses)

y_1d = y.flatten()

# Three learning rates — chosen relative to the largest eigenvalue of X^T X
alphas       = [1e-9, 1e-8, 1e-7]
alpha_labels = ['Too small  (α=1e-9)', 'Good       (α=1e-8)', 'Too large  (α=1e-7)']
colors       = ['steelblue', 'seagreen', 'tomato']
n_iters      = 200

results = {}
for alpha, label in zip(alphas, alpha_labels):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        betas, losses = gradient_descent(X_simple, y_1d, alpha, n_iters)
    results[label] = (betas, losses)

# Use the small-alpha run as the reference — it never diverges
start_loss   = results['Too small  (α=1e-9)'][1][0]
optimal_loss = float(np.sum((y_1d - X_simple @ beta_simple.flatten())**2))
ceiling      = start_loss * 1.15

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: loss vs. iteration ──────────────────────────────────────────────────
ax = axes[0]
for label, color in zip(alpha_labels, colors):
    _, losses = results[label]
    # Keep finite values as-is; replace the first diverged point with ceiling
    # and everything after with NaN so the line simply stops
    plot_losses = losses.copy()
    first_bad = next((i for i, v in enumerate(plot_losses) if not np.isfinite(v)), None)
    if first_bad is not None:
        plot_losses[first_bad]  = ceiling      # one point at the ceiling
        plot_losses[first_bad + 1:] = np.nan   # rest invisible
    plot_losses = np.clip(plot_losses, 0, ceiling)
    ax.plot(plot_losses, color=color, lw=2, label=label)
ax.axhline(optimal_loss, color='black', lw=1.2, linestyle='--',
           label='Optimal SSR (normal equations)')
ax.set_xlabel('Iteration')
ax.set_ylabel('SSR (loss)')
ax.set_title('Loss vs. Iteration for Different Learning Rates')
ax.set_ylim(0, ceiling)
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

# ── Right: gradient descent spiral on a synthetic elliptical loss surface ─────
# Uses a 2-parameter quadratic with eigenvalues [1, 10] (condition number = 10)
# to clearly illustrate the zigzagging trajectory caused by elongated contours.
ax = axes[1]

A_demo   = np.array([[5.5, 4.5], [4.5, 5.5]])   # eigenvalues 1 and 10
b_star   = np.array([2.0, 1.0])                  # true minimum
b_init   = np.array([-0.5, 0.5])                 # starting point
alpha_sp = 0.17                                   # near-stable: causes visible spiral

path  = [b_init.copy()]
beta  = b_init.copy()
for _ in range(60):
    grad = A_demo @ (beta - b_star)
    beta = beta - alpha_sp * grad
    path.append(beta.copy())
path = np.array(path)

# Contour grid
b0g = np.linspace(-0.9, 2.6, 200)
b1g = np.linspace( 0.1, 3.3, 200)
B0, B1 = np.meshgrid(b0g, b1g)
Z = np.zeros_like(B0)
for i in range(B0.shape[0]):
    for j in range(B0.shape[1]):
        db = np.array([B0[i,j], B1[i,j]]) - b_star
        Z[i,j] = 0.5 * db @ A_demo @ db

cp = ax.contourf(B0, B1, Z, levels=20, cmap='Blues')
ax.contour( B0, B1, Z, levels=20, colors='white', linewidths=0.5, alpha=0.4)
ax.plot(path[:, 0], path[:, 1], color='seagreen', lw=1.5,
        marker='o', markersize=2.5, alpha=0.8, label='GD trajectory')
ax.scatter(path[0, 0], path[0, 1], color='white',  s=80, zorder=6, label='Start')
ax.scatter(b_star[0],  b_star[1],  color='tomato', s=80, zorder=6, label=r'Minimum $\hat{\boldsymbol{\beta}}$')
ax.set_xlabel(r'$\beta_0$', fontsize=11)
ax.set_ylabel(r'$\beta_1$', fontsize=11)
ax.set_title('GD Trajectory on Elliptical Loss Contours\n(synthetic demo, condition number = 10)')
ax.legend(fontsize=9)
plt.colorbar(cp, ax=ax, label='Loss')

plt.suptitle('Gradient Descent: Learning Rates and Convergence', fontsize=12)
plt.tight_layout()
plt.show()

**Question.** In the left panel, the "too small" learning rate makes very slow progress and the "too large" one diverges. What does the loss trajectory look like for the too-large case — does it grow steadily, oscillate, or jump immediately to a very large value? What does that tell you about what's happening geometrically?

### Why Standardization Helps

The elongated contours in the loss surface — visible in both Section 10.4 and above — cause gradient descent to take inefficient zigzagging steps. The gradient points steeply across the narrow direction of the ellipse but almost horizontally along the long axis, so the algorithm overshoots in one direction while barely moving in the other.

The root cause is that the features are on very different scales. `gestation` ranges from roughly 150 to 360 days while `parity` is just 0 or 1. A one-unit change in `gestation` and a one-unit change in `parity` have completely different effects on the loss surface, stretching it into an elongated bowl.

**Standardization** rescales each feature to have mean zero and standard deviation one:

$$z = \frac{x - \bar{x}}{s_x}$$

After standardization, a one-unit change in any feature corresponds to a one-standard-deviation change in that feature. The loss surface becomes much more circular, and gradient descent converges in far fewer iterations. The cost is interpretability: the coefficients now measure the effect of a one-standard-deviation change, not a one-unit change.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_std  = scaler.fit_transform(X_sk)   # standardize all features (no intercept column)
y_std  = (y_sk - y_sk.mean()) / y_sk.std()

# Note: we do NOT add an intercept column here.
# The intercept column (all ones) is not standardized, so it dominates
# the eigenvalues of X^T X and forces alpha to be impractically small.
# sklearn's LinearRegression handles the intercept separately; for this
# gradient descent comparison we work with features only.

alpha_orig = 1e-9    # must stay below 2e-8 (stability limit for X_simple)
alpha_std  = 5e-4    # can be 50,000x larger because standardization shrinks lambda_max
n_iters_compare = 20

_, losses_orig = gradient_descent(X_simple, y_1d, alpha_orig, n_iters_compare)
_, losses_std  = gradient_descent(X_std,    y_std, alpha_std,  n_iters_compare)

# Normalize by fraction of gap closed: 0 = start, 1 = optimal
# This makes both curves comparable despite different response scales
opt_orig = float(np.sum((y_1d - X_simple @ np.linalg.inv(X_simple.T @ X_simple) @ X_simple.T @ y_1d)**2))
opt_std  = float(np.sum((y_std - X_std   @ np.linalg.inv(X_std.T   @ X_std)     @ X_std.T    @ y_std)**2))

conv_orig = 1 - (losses_orig - opt_orig) / (losses_orig[0] - opt_orig)
conv_std  = 1 - (losses_std  - opt_std)  / (losses_std[0]  - opt_std)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(conv_orig, color='tomato',   lw=2, label=f'Un-standardized  (α={alpha_orig:.0e},  stability limit ≈ 2e-8)')
ax.plot(conv_std,  color='seagreen', lw=2, label=f'Standardized     (α={alpha_std:.0e},  stability limit ≈ 1.5e-3)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Fraction of optimal gap closed  (0 = start, 1 = converged)')
ax.set_title('Standardization Allows a Much Larger Learning Rate\n'
             r'$\alpha_\mathrm{std}$ is $\approx$ 500,000× larger than $\alpha_\mathrm{orig}$')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

**Question.** The standardized model needs a much larger learning rate ($\alpha = 0.05$) than the un-standardized one ($\alpha = 10^{-9}$). Why? What would happen if you tried $\alpha = 0.05$ on the un-standardized data?

**Question.** After standardizing $\mathbf{X}$ and fitting a model, the coefficients are no longer in the original units. If the coefficient on `gestation` in the standardized model is 0.43, what does that mean? How does it compare to the 0.46 oz/day coefficient from the un-standardized model?

## 10.6 Logistic Regression

### Classification Problems

So far we have been predicting a *numerical* outcome — birth weight in ounces. But many real problems call for predicting a *categorical* outcome: did the mother smoke or not? Will a transaction be fraudulent? Does a tumor appear benign or malignant?

These are **classification** problems. The target variable $y$ takes discrete values — typically 0 or 1. We would like a model that predicts the *probability* that $y = 1$ given the features $x_1, \ldots, x_p$.

### Why Not Linear Regression?

A linear regression model could predict a number, but that number might be greater than 1 or less than 0 — not a valid probability. More fundamentally, the SSR loss function is no longer appropriate: minimizing squared error between a 0/1 outcome and a probability estimate produces a poor classifier with unstable coefficients.

### The Logistic Model

**Logistic regression** solves this by transforming the linear predictor through the **sigmoid function**:

$$\hat{p}(x_1, \ldots, x_p) = \frac{e^{\beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p}}{1 + e^{\beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p}} = \sigma(\mathbf{x}^T\boldsymbol{\beta})$$

The sigmoid function maps any real number to the interval $(0, 1)$, so $\hat{p}$ is always a valid probability. We predict class 1 when $\hat{p} > 0.5$ and class 0 otherwise.

In [ ]:
# Visualize the sigmoid function
z   = np.linspace(-8, 8, 300)
sig = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(z, sig, color='steelblue', lw=2.5)
ax.axhline(0.5, color='tomato', lw=1.2, linestyle='--', label='Decision boundary: p = 0.5')
ax.axhline(0,   color='#aaa',   lw=0.8, linestyle=':')
ax.axhline(1,   color='#aaa',   lw=0.8, linestyle=':')
ax.axvline(0,   color='#ccc',   lw=0.8)
ax.fill_between(z, sig, 0.5, where=(sig > 0.5), alpha=0.12, color='seagreen',
                label='Predict class 1 (smoke=1)')
ax.fill_between(z, sig, 0.5, where=(sig < 0.5), alpha=0.12, color='tomato',
                label='Predict class 0 (smoke=0)')
ax.set_xlabel(r'Linear predictor $z = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$', fontsize=10)
ax.set_ylabel(r'$\hat{p} = \sigma(z)$', fontsize=10)
ax.set_title(r'Sigmoid Function: Maps $\mathbb{R}$ to $(0, 1)$', fontsize=11)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

**Question.** The sigmoid is nonlinear in $\boldsymbol{\beta}$. Why does this mean we can't solve for $\boldsymbol{\hat{\beta}}$ using the normal equations? *(Hint: setting $\nabla f = \mathbf{0}$ and rearranging requires linearity — the same algebra that gave us $(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ doesn't work when there's a sigmoid wrapped around the linear predictor.)*

### Training with scikit-learn

Since there is no closed-form solution, sklearn trains logistic regression using gradient descent (specifically a variant called L-BFGS or liblinear, depending on the `solver` argument). The `.n_iter_` attribute records how many gradient descent iterations it took to converge.

We'll predict `smoke` (whether the mother smoked during pregnancy) from the other variables. First we need to drop rows where `smoke` has a NaN value — here we use `dropna()` so the target is always a clean 0 or 1.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Drop rows where smoke is missing; fill remaining NaNs with column means
babies_clf = pd.read_csv(url).dropna(subset=['smoke'])
babies_clf = babies_clf.fillna(babies_clf.mean(numeric_only=True))

clf_features = ['gestation', 'parity', 'age', 'height', 'weight', 'bwt']
X_clf = babies_clf[clf_features].to_numpy()
y_clf = babies_clf['smoke'].to_numpy().astype(int)

print(f'Dataset: {X_clf.shape[0]} observations, {X_clf.shape[1]} features')
print(f'Class counts:  smoke=0 → {(y_clf==0).sum()},  smoke=1 → {(y_clf==1).sum()}')

In [ ]:
# Train logistic regression — no standardization first
babies_logistic = LogisticRegression(solver='liblinear', max_iter=1000).fit(X_clf, y_clf) # Try "solver = liblinear!"

print(f'Converged in {babies_logistic.n_iter_[0]} iterations')
print(f'Accuracy: {babies_logistic.score(X_clf, y_clf):.4f}')

In [ ]:
# Now standardize and retrain
scaler_clf    = StandardScaler()
X_clf_std     = scaler_clf.fit_transform(X_clf)

babies_logistic_std = LogisticRegression(solver='liblinear', max_iter=1000).fit(X_clf_std, y_clf)

print(f'Standardized — converged in {babies_logistic_std.n_iter_[0]} iterations')
print(f'Accuracy: {babies_logistic_std.score(X_clf_std, y_clf):.4f}')

**Question.** The standardized model converges in fewer iterations than the un-standardized one. This is the same phenomenon we saw with gradient descent in Section 10.5 — but now it's sklearn doing the work internally. What does the reduction in iterations tell you about the shape of the logistic regression loss surface before and after standardization?

**Note.** The accuracy is the same (or very close) with and without standardization. The coefficients, however, will be different — different in scale but equivalent in predictive power. 

In [ ]:
# Inspect coefficients and make a prediction
print('Logistic regression coefficients (standardized features):')
for name, val in zip(clf_features, babies_logistic_std.coef_[0]):
    print(f'  {name:<12}: {val:+.4f}')
print(f'  {"intercept":<12}: {babies_logistic_std.intercept_[0]:+.4f}')

print()

# Predict smoking probability for a specific mother
# gestation=280, parity=0, age=27, height=64, weight=120, bwt=120
example = np.array([[280, 0, 27, 64, 120, 120]])
example_std = scaler_clf.transform(example)

p_hat  = babies_logistic_std.predict_proba(example_std)[0]
y_pred = babies_logistic_std.predict(example_std)[0]

print(f'Predicted probability of smoking: {p_hat[1]:.4f}')
print(f'Predicted class (smoke):          {y_pred}  ({"smoker" if y_pred == 1 else "non-smoker"})')

## 10.7 Evaluating a Classifier: The Confusion Matrix

### Why Accuracy Alone is Not Enough

You might be tempted to evaluate a classifier the same way we evaluated a regression model — compute a single number that summarizes performance. For regression, that number was RMSE: the average size of our prediction errors. But for classification, **RMSE is not an appropriate metric**. The response variable is 0 or 1, not a continuous quantity, so the geometry of "how far off" we were does not apply. A predicted probability of 0.49 and a predicted probability of 0.01 both produce the same class label (0), even though they represent very different levels of confidence.

The most natural summary for a classifier is **accuracy** — the fraction of observations that were classified correctly:

$$\text{Accuracy} = \frac{\text{number of correct predictions}}{n}$$

Accuracy sounds appealing, but it can be deeply misleading. Suppose only 5% of patients in a medical dataset have a disease. A model that always predicts "healthy" achieves 95% accuracy — better than many trained classifiers — while being completely useless. It never detects the disease at all.

The deeper problem is that accuracy treats all errors as equivalent. In most real settings, the two types of errors have very different consequences:

- **False positive:** predicting class 1 when the true label is 0 — incorrectly flagging a non-smoker as a smoker.
- **False negative:** predicting class 0 when the true label is 1 — missing a smoker entirely.

Whether a false positive or a false negative is more harmful depends entirely on the domain. In medical screening, missing a disease (false negative) may be far worse than a follow-up test (false positive). In spam detection, wrongly blocking a legitimate email (false positive) may be worse than letting one spam message through.

### The Confusion Matrix

The **confusion matrix** lays out all four possible outcomes in a $2 \times 2$ table. Rows index the **true** class; columns index the **predicted** class:

|  | Predicted 0 | Predicted 1 |
|:---:|:---:|:---:|
| **True 0** | True Negative (TN) | False Positive (FP) |
| **True 1** | False Negative (FN) | True Positive (TP) |

- **True Negative (TN):** Correctly predicted non-smoker.
- **False Positive (FP):** Predicted smoker, actually a non-smoker. *(Type I error)*
- **False Negative (FN):** Predicted non-smoker, actually a smoker. *(Type II error)*
- **True Positive (TP):** Correctly predicted smoker.

Accuracy is just the diagonal sum: $\text{Accuracy} = (TN + TP) / n$. But the off-diagonal cells — the errors — tell you *how* the model is failing, not just *that* it is failing.

Two additional metrics are often reported alongside accuracy:

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN}$$

**Precision** answers: of all the observations the model *predicted* as smokers, what fraction actually were? **Recall** answers: of all the *actual* smokers, what fraction did the model catch? A model can be tuned to favor one over the other by adjusting the decision threshold away from 0.5.

In [ ]:
# Run this cell without modification
from sklearn import metrics

y_pred_std = babies_logistic_std.predict(X_clf_std)

cm  = metrics.confusion_matrix(y_clf, y_pred_std)
acc = metrics.accuracy_score(y_clf, y_pred_std)
TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

print('Confusion matrix (rows = true label, columns = predicted label):')
print(f'  {"":18} Predicted 0   Predicted 1')
print(f'  True 0  (non-smoker)   {TN:>6}        {FP:>6}   <- False Positives')
print(f'  True 1  (smoker)       {FN:>6}        {TP:>6}   <- True Positives')
print()
print(f'Accuracy:  {acc:.4f}   ({TN+TP} correct out of {len(y_clf)})')
print(f'Baseline (always predict non-smoker): {(y_clf==0).sum()/len(y_clf):.4f}')
print()
if TP + FP > 0:
    precision = TP / (TP + FP)
    print(f'Precision: {precision:.4f}   (of predicted smokers, {precision:.1%} were actually smokers)')
recall = TP / (TP + FN)
print(f'Recall:    {recall:.4f}   (of actual smokers, {recall:.1%} were correctly identified)')
print()
print(f'False positives: {FP}  (non-smokers flagged as smokers)')
print(f'False negatives: {FN}  (smokers missed entirely)')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: confusion matrix heatmap
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted\nnon-smoker', 'Predicted\nsmoker'], fontsize=10)
ax.set_yticklabels(['True\nnon-smoker', 'True\nsmoker'], fontsize=10)
for i in range(2):
    for j in range(2):
        label = {(0,0): f'TN\n{cm[0,0]}', (0,1): f'FP\n{cm[0,1]}',
                 (1,0): f'FN\n{cm[1,0]}', (1,1): f'TP\n{cm[1,1]}'}[(i,j)]
        color = 'white' if cm[i,j] > cm.max()/2 else 'black'
        ax.text(j, i, label, ha='center', va='center', fontsize=13,
                fontweight='bold', color=color)
ax.set_title(f'Confusion Matrix  (Accuracy = {acc:.4f})', fontsize=11)
ax.set_xlabel('Predicted label', fontsize=10)
ax.set_ylabel('True label', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)

# Right: predicted probability distribution by true class
ax = axes[1]
y_prob_std = babies_logistic_std.predict_proba(X_clf_std)[:, 1]
ax.hist(y_prob_std[y_clf == 0], bins=30, alpha=0.6, color='steelblue',
        label='True non-smoker (y=0)', density=True)
ax.hist(y_prob_std[y_clf == 1], bins=30, alpha=0.6, color='tomato',
        label='True smoker (y=1)', density=True)
ax.axvline(0.5, color='black', lw=1.5, linestyle='--', label='Decision threshold (0.5)')
ax.set_xlabel(r'Predicted probability of smoking $\hat{p}$', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('Predicted Probabilities by True Class', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Logistic Regression — Classifier Evaluation', fontsize=12)
plt.tight_layout()
plt.show()

**Question.** Look at the predicted probability histogram on the right. The two distributions — smokers and non-smokers — overlap substantially in the middle. What does this overlap tell you about the difficulty of the classification problem? What would the histogram look like for a perfect classifier?

**Question.** The confusion matrix shows both false positives and false negatives. Suppose a public health researcher is using this model to identify mothers who might benefit from a smoking cessation program. Which type of error is more costly in this context — a false positive or a false negative — and why? How would you adjust the decision threshold to reduce that error?

**Question.** The model's accuracy may be only slightly better than the baseline of always predicting the majority class. Why is comparing to the baseline important, and what would it mean if the model's accuracy were *lower* than the baseline?

**Looking ahead.** This lecture completes our tour of supervised learning foundations: least squares, the normal equations, regression metrics, the sklearn API, gradient descent, and classification with logistic regression. You now have all the tools to complete your final project.